In [ ]:
# =============================
# RideWise Customer Churn EDA
# =============================

import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_style("whitegrid")

BASE_DIR = os.path.dirname(os.getcwd())
DB_PATH = os.path.join(BASE_DIR, "db", "ridewise.sqlite")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM user_features", conn)
conn.close()

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

df.info()

df.describe()

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
churn_rate = df["churn_30d"].mean()
print("Churn rate:", churn_rate)

sns.countplot(x="churn_30d", data=df)
plt.title("Churn Distribution")
plt.show()

# Trips vs Churn

In [ ]:
sns.boxplot(x="churn_30d", y="trips_count", data=df)
plt.title("Trips Count vs Churn")
plt.show()

# Spend vs Churn

In [ ]:
sns.boxplot(x="churn_30d", y="spend_per_trip", data=df)
plt.title("Spend per Trip vs Churn")
plt.show()

# Recency vs Churn

In [ ]:
sns.boxplot(x="churn_30d", y="recency_days", data=df)
plt.title("Recency vs Churn")
plt.show()



# Segment Analysis

In [ ]:
seg_churn = df.groupby("segment")["churn_30d"].mean().sort_values(ascending=False)

seg_churn.plot(kind="bar")
plt.title("Churn Rate by Segment")
plt.show()

# City Analysis

In [ ]:
city_churn = df.groupby("city")["churn_30d"].mean().sort_values(ascending=False)

city_churn.head(10).plot(kind="bar")
plt.title("Top Cities by Churn Rate")
plt.show()

# Loyalty Analysis

In [ ]:
loyalty_churn = df.groupby("loyalty_status")["churn_30d"].mean()

loyalty_churn.plot(kind="bar")
plt.title("Churn by Loyalty Status")
plt.show()

# Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include="number").columns

plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

Distribution Plots

In [ ]:
cols = ["trips_count", "recency_days", "spend_per_trip"]

for c in cols:
    sns.histplot(df[c], bins=30)
    plt.title(c)
    plt.show()

Key EDA Findings

- Overall churn rate: 19.9%
- High recency strongly linked to churn
- Low trips users churn more
- Weekend users have highest churn
- Non-loyal users churn more

Feature Importance Distribution

In [ ]:
import joblib

MODEL_PATH = os.path.join(BASE_DIR, "models", "random_forest.joblib")

model = joblib.load(MODEL_PATH)

model

In [ ]:
import joblib

MODEL_PATH = os.path.join(BASE_DIR, "models", "random_forest.joblib")

model = joblib.load(MODEL_PATH)

model

Plot Top Features

In [ ]:
top_n = 15
top_fi = fi.head(top_n)

plt.figure(figsize=(8,5))
sns.barplot(data=top_fi, y="feature", x="importance")
plt.title("Top Feature Importances (Random Forest)")
plt.show()

Importance Distribution

In [ ]:
sns.histplot(fi["importance"], bins=30)
plt.title("Distribution of Feature Importances")
plt.show()

Cumulative Importance

In [ ]:
fi["cumulative"] = fi["importance"].cumsum()

plt.plot(fi["cumulative"].values)
plt.title("Cumulative Feature Importance")
plt.xlabel("Features")
plt.ylabel("Cumulative Importance")
plt.show()

Feature Importance Insights

Recency-related features are strongest predictors of churn.

Trip frequency and engagement metrics contribute significantly.

Spending behavior also influences churn likelihood.

Demographic and geographic variables show lower importance.

A small subset of features explains most model variance.

These results align with EDA findings that inactivity and low usage drive churn.